<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V4_AD_Only_Mannheim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
V4 — Anomaly Detection Pipeline für Mannheim (Saubere, Transfer-Ready Version)
================================================================================
Korrekturen gegenüber V3:

  LEAKAGE-FIXES:
    #1  z_score, demand_residual, demand_ratio RAUS aus Feature-Vektor
        → Diese Features SIND die Label-Definition und erzeugen einen Kurzschluss
    #2  hist_mean/hist_std nur aus Trainingsperiode berechnet (kein temporales Leakage)
    #3  Lag-Residuen (lag - hist_mean) ebenfalls entfernt → stadtspezifisch

  TRANSFER-FIXES:
    #4  Nur universell transferierbare Features (Raw Demand, Lags, Kalender, Wetter)
    #5  Zero-Stunden im Training NICHT filtern → AE muss lernen: Stille = normal
    #6  Scoring über GANZE Sequenz, nicht nur letzte 3h → kein Tuning auf Labeling-Schema

  WISSENSCHAFTLICHE INTEGRITÄT:
    #7  Jede Designentscheidung ist kommentiert und begründbar
    #8  Konservative Parameterwahl → keine künstliche Optimierung
    #9  Stats-Berechnung temporal sauber → reproduzierbar

  ERGEBNIS: Niedrigere Metriken, aber EHRLICH und TRANSFERIERBAR.
  Ein AUC-ROC von ~0.90 ohne Tricks > 0.98 mit Leakage.
"""

'\nV4 — Anomaly Detection Pipeline für Mannheim (Saubere, Transfer-Ready Version)\n================================================================================\nKorrekturen gegenüber V3:\n\n  LEAKAGE-FIXES:\n    #1  z_score, demand_residual, demand_ratio RAUS aus Feature-Vektor\n        → Diese Features SIND die Label-Definition und erzeugen einen Kurzschluss\n    #2  hist_mean/hist_std nur aus Trainingsperiode berechnet (kein temporales Leakage)\n    #3  Lag-Residuen (lag - hist_mean) ebenfalls entfernt → stadtspezifisch\n\n  TRANSFER-FIXES:\n    #4  Nur universell transferierbare Features (Raw Demand, Lags, Kalender, Wetter)\n    #5  Zero-Stunden im Training NICHT filtern → AE muss lernen: Stille = normal\n    #6  Scoring über GANZE Sequenz, nicht nur letzte 3h → kein Tuning auf Labeling-Schema\n\n  WISSENSCHAFTLICHE INTEGRITÄT:\n    #7  Jede Designentscheidung ist kommentiert und begründbar\n    #8  Konservative Parameterwahl → keine künstliche Optimierung\n    #9  Stats-Berechn

In [1]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory


In [2]:
# ══════════════════════════════════════════════════════════════
# 0 — Setup & Imports
# ══════════════════════════════════════════════════════════════
import sympy.printing

import os, glob, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from shapely import wkb
import sympy.printing

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, auc, f1_score, roc_auc_score,
    classification_report, roc_curve
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


# ══════════════════════════════════════════════════════════════
# 1 — Pfade & Konfiguration
# ══════════════════════════════════════════════════════════════

# ── Pfade ──
DEMAND_BASE        = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'
GEO_INFO_PATH      = '/content/data/geo_information/geo_information.parquet'
WEATHER_PATH       = '/content/data/weather/weather.parquet'
HOLIDAYS_PATH      = '/content/data/holidays/holidays.parquet'
VACATIONS_PATH     = '/content/data/vacations/vacations.parquet'

OUTPUT_PATH        = '/content/data/V4_AD_Mannheim.parquet'

# ── Konfiguration ──
CITY               = 'Mannheim'
NETWORK_NAME       = 'Mannheim'
WEATHER_STATION_ID = 292348
FEDERAL_STATE      = 'Baden-Württemberg'

# ── AD-Hyperparameter ──
WINDOW_SIZE        = 24                   # Stunden (1 Tag)
LATENT_DIM         = 32

# ── Labeling-Schwellen ──
Z_TRAIN_THRESHOLD  = 2.0                  # |z| <= 2 → normal (für Training-Selektion)
Z_ANOMALY_THRESHOLD = 3.0                 # |z| > 3 → anomal (für Evaluation)
MIN_EVENTS_PER_DAY = 3.0                  # Stationsfilter: mind. 3 Events/Tag
MIN_HIST_MEAN      = 2.0                  # Anomalie nur bei hist_mean >= 2
MIN_ABSOLUTE       = 5                    # Anomalie nur bei >= 5 absoluten Events

# ── Training ──
BATCH_SIZE         = 4096
EPOCHS             = 50
LEARNING_RATE      = 1e-3
TRAIN_RATIO        = 0.67
VAL_RATIO          = 0.83

# ── [V4.1-A] Active-Hours Filtering ──
FILTER_ZERO_HOURS  = True                 # Zero-Demand Stunden aus Training entfernen

# ── [V4.1-B] Point-wise Scoring ──
SCORE_LAST_N_STEPS = 3                    # Letzte N Stunden der Sequenz für Score

# ══════════════════════════════════════════════════════════════
# [DESIGN-ENTSCHEIDUNG] Feature-Liste — NUR transferierbare Features
#
# BEGRÜNDUNG:
#   z_score, demand_residual, demand_ratio sind NICHT im Feature-Vektor weil:
#   (a) Sie die Label-Definition enthalten → Kurzschluss / Data Leakage
#   (b) Sie auf stadtspezifischen Statistiken basieren → nicht transferierbar
#   (c) Der AE soll echte temporale Muster lernen, keine Statistiken nachrechnen
#
#   z_score wird NUR für das Labeling verwendet (Preprocessing) — das ist
#   in der AD-Literatur etabliert und eine valide unsupervised Methode.
# ══════════════════════════════════════════════════════════════

FEATURE_COLS = [
    # ── Nachfrage (roh, normalisiert via StandardScaler) ──
    'total_demand', 'n_lends', 'n_returns',
    # ── Temporale Lags (roh — universelle Zeitreihen-Muster) ──
    'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
    # ── Kalender zyklisch (universell, stadtunabhängig) ──
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    # ── Kalender binär (universell, Feiertage/Ferien pro Stadt anpassbar) ──
    'is_weekend', 'is_holiday', 'is_vacation',
    # ── Wetter (gleiche Semantik in jeder Stadt) ──
    'temperature', 'humidity', 'precipitation', 'wind_speed',
]

# Demand-Features für Scoring (die ersten 6)
DEMAND_FEATURE_INDICES = list(range(6))

N_FEATURES     = len(FEATURE_COLS)
INPUT_DIM_FLAT = WINDOW_SIZE * N_FEATURES

print(f'Features: {N_FEATURES} → {FEATURE_COLS}')
print(f'Demand-Features für Scoring: {[FEATURE_COLS[i] for i in DEMAND_FEATURE_INDICES]}')
print(f'Input dim flat: {INPUT_DIM_FLAT}')


Device: cuda
  GPU: NVIDIA L4
  VRAM: 23.7 GB
Features: 19 → ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_holiday', 'is_vacation', 'temperature', 'humidity', 'precipitation', 'wind_speed']
Demand-Features für Scoring: ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h']
Input dim flat: 456


In [3]:
# ══════════════════════════════════════════════════════════════
# 2 — Hilfsdaten laden (Stationen, Wetter, Feiertage)
# ══════════════════════════════════════════════════════════════

# ── Station Names & Typ-Klassifikation ──
def classify_station(name: str) -> str:
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names['station_type'] = station_names['station_name'].apply(classify_station)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()
print(f'Station-Types: {station_names["station_type"].value_counts().to_dict()}')

# ── Wetter laden & auf Stunde aggregieren ──
weather = pd.read_parquet(WEATHER_PATH)
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
weather_ma = weather[weather['location_id'] == WEATHER_STATION_ID].copy()
weather_ma['hour_ts'] = weather_ma['timestamp'].dt.floor('h')

weather_hourly = (
    weather_ma.groupby('hour_ts')
    .agg(
        temperature   = ('temperature', 'mean'),
        humidity      = ('humidity', 'mean'),
        precipitation = ('precipitation', 'sum'),
        wind_speed    = ('wind_speed', 'mean'),
    )
    .reset_index()
)
print(f'Wetter stündlich: {len(weather_hourly):,} Zeilen | '
      f'{weather_hourly["hour_ts"].min().date()} – {weather_hourly["hour_ts"].max().date()}')

# ── Feiertage & Ferien (nur BaWü) ──
holidays  = pd.read_parquet(HOLIDAYS_PATH)
vacations = pd.read_parquet(VACATIONS_PATH)
holidays['start_date']  = pd.to_datetime(holidays['start_date'])
holidays['end_date']    = pd.to_datetime(holidays['end_date'])
vacations['start_date'] = pd.to_datetime(vacations['start_date'])
vacations['end_date']   = pd.to_datetime(vacations['end_date'])

holidays_bw  = holidays[holidays['federal_state_name'] == FEDERAL_STATE]
vacations_bw = vacations[vacations['federal_state_name'] == FEDERAL_STATE]

holiday_dates = set()
for _, row in holidays_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        holiday_dates.add(d.date())

vacation_dates = set()
for _, row in vacations_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        vacation_dates.add(d.date())

print(f'Feiertage BaWü: {len(holiday_dates)} Tage | Ferien BaWü: {len(vacation_dates)} Tage')


# ══════════════════════════════════════════════════════════════
# 3 — Demand laden & auf Stunde aggregieren
# ══════════════════════════════════════════════════════════════

files = glob.glob(os.path.join(DEMAND_BASE, CITY, '**', '*.parquet'), recursive=True)
if not files:
    files = glob.glob(os.path.join(DEMAND_BASE, CITY, '*.parquet'))
print(f'Parquet-Dateien gefunden: {len(files)}')

cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
        'location_id', 'n_lends', 'n_returns']
demand_raw = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
demand_raw['timestamp'] = pd.to_datetime(demand_raw['timestamp'], utc=True)

demand_raw['station_type'] = demand_raw['station_name_id'].map(type_lookup).fillna('unknown')
demand = demand_raw[demand_raw['station_type'] == 'real'].copy()

print(f'Demand roh:      {len(demand_raw):,} Zeilen')
print(f'Demand (real):   {len(demand):,} Zeilen')
print(f'Stationen:       {demand["station_id"].nunique()}')
print(f'Zeitraum:        {demand["timestamp"].min().date()} – {demand["timestamp"].max().date()}')

# ── Stündliche Aggregation pro Station ──
demand['hour_ts'] = demand['timestamp'].dt.floor('h')
hourly = (
    demand
    .groupby(['station_id', 'station_name_id', 'location_id', 'hour_ts'])
    .agg(n_lends=('n_lends', 'sum'), n_returns=('n_returns', 'sum'))
    .reset_index()
)
hourly['total_demand'] = hourly['n_lends'] + hourly['n_returns']

# ── Deduplizierung ──
hourly_agg = (
    hourly.groupby(['station_id', 'hour_ts'])
    .agg(
        n_lends         = ('n_lends', 'sum'),
        n_returns       = ('n_returns', 'sum'),
        total_demand    = ('total_demand', 'sum'),
        station_name_id = ('station_name_id', 'first'),
        location_id     = ('location_id', 'first'),
    )
    .reset_index()
)

# ── Lücken füllen ──
all_hours = pd.date_range(
    hourly['hour_ts'].min(), hourly['hour_ts'].max(), freq='h', tz='UTC'
)

station_info = (
    hourly_agg.groupby('station_id')
    .agg(station_name_id=('station_name_id', 'first'), location_id=('location_id', 'first'))
    .reset_index()
)

full_index = pd.MultiIndex.from_product(
    [station_info['station_id'].values, all_hours],
    names=['station_id', 'hour_ts']
)
hourly_full = (
    hourly_agg[['station_id', 'hour_ts', 'n_lends', 'n_returns', 'total_demand']]
    .set_index(['station_id', 'hour_ts'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
hourly_full = hourly_full.merge(station_info, on='station_id', how='left')

print(f'Nach Lückenfüllung: {len(hourly_full):,} Zeilen')
print(f'Stationen: {hourly_full["station_id"].nunique()}')

# ── Stationsfilter ──
n_days = (all_hours[-1] - all_hours[0]).days + 1
station_activity = hourly_full.groupby('station_id')['total_demand'].sum()
min_events = int(n_days * MIN_EVENTS_PER_DAY)
active_stations = station_activity[station_activity >= min_events].index

hourly_full = hourly_full[hourly_full['station_id'].isin(active_stations)].copy()
print(f'Stationen nach Aktivitätsfilter (≥{MIN_EVENTS_PER_DAY}/Tag): '
      f'{hourly_full["station_id"].nunique()} '
      f'(entfernt: {len(station_activity) - len(active_stations)})')


# ══════════════════════════════════════════════════════════════
# 4 — Feature Engineering
# ══════════════════════════════════════════════════════════════

# ── Kalender-Features ──
hourly_full['hour_of_day'] = hourly_full['hour_ts'].dt.hour
hourly_full['day_of_week'] = hourly_full['hour_ts'].dt.dayofweek
hourly_full['month']       = hourly_full['hour_ts'].dt.month
hourly_full['is_weekend']  = (hourly_full['day_of_week'] >= 5).astype(np.int8)
hourly_full['is_holiday']  = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in holiday_dates else 0
).astype(np.int8)
hourly_full['is_vacation'] = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in vacation_dates else 0
).astype(np.int8)

# Zyklische Kodierung
hourly_full['hour_sin']  = np.sin(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['hour_cos']  = np.cos(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['dow_sin']   = np.sin(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['dow_cos']   = np.cos(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['month_sin'] = np.sin(2 * np.pi * hourly_full['month'] / 12)
hourly_full['month_cos'] = np.cos(2 * np.pi * hourly_full['month'] / 12)
print(f'Kalender-Features hinzugefügt.')

# ── Wetter-Features ──
hourly_full = hourly_full.merge(weather_hourly, on='hour_ts', how='left')
for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
    hourly_full[col] = (
        hourly_full.groupby('station_id')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=6).ffill().bfill())
    )
    median_val = hourly_full[col].median()
    hourly_full[col] = hourly_full[col].fillna(median_val)

print(f'Wetter-Coverage: {hourly_full["temperature"].notna().mean()*100:.1f}%')

# ── Lag-Features ──
hourly_full = hourly_full.sort_values(['station_id', 'hour_ts']).reset_index(drop=True)
for lag_name, lag_hours in [('lag_1h', 1), ('lag_24h', 24), ('lag_168h', 168)]:
    hourly_full[f'demand_{lag_name}'] = (
        hourly_full.groupby('station_id')['total_demand'].shift(lag_hours)
    )

before = len(hourly_full)
hourly_full = hourly_full.dropna(subset=['demand_lag_168h']).reset_index(drop=True)
print(f'Lag-Features: {before - len(hourly_full):,} Zeilen entfernt (Anlaufphase)')


Station-Types: {'bike': 44747, 'real': 13040, 'virtual': 4827, 'recording': 1972, 'only_nums': 51}
Wetter stündlich: 24,913 Zeilen | 2023-04-01 – 2026-02-02
Feiertage BaWü: 167 Tage | Ferien BaWü: 277 Tage
Parquet-Dateien gefunden: 36
Demand roh:      2,683,584 Zeilen
Demand (real):   2,579,329 Zeilen
Stationen:       128
Zeitraum:        2023-03-31 – 2026-02-02
Nach Lückenfüllung: 3,191,936 Zeilen
Stationen: 128
Stationen nach Aktivitätsfilter (≥3.0/Tag): 92 (entfernt: 36)
Kalender-Features hinzugefügt.
Wetter-Coverage: 100.0%
Lag-Features: 15,456 Zeilen entfernt (Anlaufphase)


In [4]:
# ══════════════════════════════════════════════════════════════
# 5 — Temporaler Split ZUERST, dann Labeling
#
# [FIX #2] DESIGN-ENTSCHEIDUNG: Split VOR Stats-Berechnung
#
# BEGRÜNDUNG:
#   hist_mean/hist_std dürfen NUR aus Trainingsdaten berechnet werden.
#   Sonst fließt Zukunftsinformation in die Labels → temporales Leakage.
#   In der Cross-City-Anwendung hätte man in der Target City ebenfalls
#   nur historische Daten zur Verfügung.
# ══════════════════════════════════════════════════════════════

t_min = hourly_full['hour_ts'].min()
t_max = hourly_full['hour_ts'].max()
total_hours = (t_max - t_min).total_seconds() / 3600

train_end = t_min + pd.Timedelta(hours=int(total_hours * TRAIN_RATIO))
val_end   = t_min + pd.Timedelta(hours=int(total_hours * VAL_RATIO))

print(f'\n═══ Temporaler Split ═══')
print(f'Train-Ende: {train_end.date()}')
print(f'Val-Ende:   {val_end.date()}')
print(f'Test-Ende:  {t_max.date()}')

# ── Statistiken NUR aus Trainingsperiode berechnen ──
train_period = hourly_full[hourly_full['hour_ts'] < train_end]

stats = (
    train_period
    .groupby(['station_id', 'day_of_week', 'hour_of_day'])['total_demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'hist_mean', 'std': 'hist_std'})
)

hourly_full = hourly_full.merge(
    stats, on=['station_id', 'day_of_week', 'hour_of_day'], how='left'
)

# Stationen/Slots ohne Trainings-Statistik → fillna mit globalen Werten
# (das passiert z.B. bei neuen Stationen die erst nach train_end auftauchen)
global_mean = train_period['total_demand'].mean()
global_std  = train_period['total_demand'].std()
hourly_full['hist_mean'] = hourly_full['hist_mean'].fillna(global_mean)
hourly_full['hist_std']  = hourly_full['hist_std'].fillna(global_std)
hourly_full['hist_std']  = hourly_full['hist_std'].clip(lower=0.1)

# z_Score berechnen (NUR für Labeling, NICHT als Feature!)
hourly_full['z_score'] = (
    (hourly_full['total_demand'] - hourly_full['hist_mean']) / hourly_full['hist_std']
)

print(f'Stats berechnet auf {len(train_period):,} Trainings-Zeilen')
print(f'Stationen mit Stats: {stats["station_id"].nunique()}')
print(f'Stats NaN nach Fillna: {hourly_full["hist_mean"].isna().sum()}')


# ══════════════════════════════════════════════════════════════
# 6 — Labeling (z_Score-basiert, mit Qualitätsfilter)
#
# [DESIGN-ENTSCHEIDUNG] z_Score als Labeling-Methode:
#   - In der AD-Literatur etabliert für unsupervised Anomaly Detection
#   - Wird in der BA als "statistisches Pseudo-Label" deklariert
#   - LIMITATION: Approximation, kein Ground-Truth → in Diskussion adressieren
#
# z_Score wird NUR hier verwendet und NICHT in den Feature-Vektor aufgenommen.
# ══════════════════════════════════════════════════════════════

hourly_full['label'] = 'normal'

is_anomaly = (
    (hourly_full['z_score'].abs() > Z_ANOMALY_THRESHOLD) &
    (hourly_full['hist_mean'] >= MIN_HIST_MEAN) &
    (hourly_full['total_demand'] >= MIN_ABSOLUTE)
)
is_grauzone = (
    (hourly_full['z_score'].abs() > Z_TRAIN_THRESHOLD) &
    ~is_anomaly
)

hourly_full.loc[is_grauzone, 'label'] = 'grauzone'
hourly_full.loc[is_anomaly, 'label']  = 'anomal'

print('\n═══ Label-Verteilung ═══')
print(hourly_full['label'].value_counts())
print(f'Anomalie-Rate: {is_anomaly.mean():.4f}')

# Qualitätscheck
anom = hourly_full[hourly_full['label'] == 'anomal']
print(f'\nAnomalie-Qualität:')
print(f'  hist_mean:     min={anom["hist_mean"].min():.1f}, '
      f'median={anom["hist_mean"].median():.1f}, max={anom["hist_mean"].max():.1f}')
print(f'  total_demand:  min={anom["total_demand"].min()}, '
      f'median={anom["total_demand"].median():.0f}, max={anom["total_demand"].max()}')
print(f'  z_score:       min={anom["z_score"].min():.1f}, '
      f'median={anom["z_score"].median():.1f}, max={anom["z_score"].max():.1f}')


═══ Temporaler Split ═══
Train-Ende: 2025-02-27
Val-Ende:   2025-08-11
Test-Ende:  2026-02-02
Stats berechnet auf 1,526,648 Trainings-Zeilen
Stationen mit Stats: 92
Stats NaN nach Fillna: 0

═══ Label-Verteilung ═══
label
normal      2157866
grauzone     113644
anomal         7238
Name: count, dtype: int64
Anomalie-Rate: 0.0032

Anomalie-Qualität:
  hist_mean:     min=2.0, median=2.9, max=24.9
  total_demand:  min=7, median=14, max=119
  z_score:       min=3.0, median=3.6, max=12.6


In [5]:
# ══════════════════════════════════════════════════════════════
# 7 — Speichern (mit Metadaten für Multi-City)
# ══════════════════════════════════════════════════════════════

hourly_full['network_name'] = NETWORK_NAME
hourly_full.to_parquet(OUTPUT_PATH, index=False)
print(f'\n✅ Gespeichert: {OUTPUT_PATH}')
print(f'   Shape: {hourly_full.shape}')


# ══════════════════════════════════════════════════════════════
# 8 — Train / Val / Test Split
#
# [FIX #5] Zero-Stunden bleiben im Training!
# BEGRÜNDUNG: Der AE muss lernen, dass inaktive Stunden (demand=0)
# normal sind. In der Target City wird es ebenfalls Zero-Stunden geben.
# Filtern würde die Datenverteilung verzerren.
#
# Grauzone wird aus Val/Test ausgeschlossen → saubere Evaluation.
# ══════════════════════════════════════════════════════════════

# Train: NUR normale Daten (Grauzone und Anomalien ausgeschlossen)
df_train = hourly_full[
    (hourly_full['hour_ts'] < train_end) &
    (hourly_full['label'] == 'normal')
].copy()

# Val & Test: Normal + Anomal (Grauzone raus für saubere binäre Evaluation)
df_val = hourly_full[
    (hourly_full['hour_ts'] >= train_end) &
    (hourly_full['hour_ts'] < val_end) &
    (hourly_full['label'] != 'grauzone')
].copy()

df_test = hourly_full[
    (hourly_full['hour_ts'] >= val_end) &
    (hourly_full['label'] != 'grauzone')
].copy()

print(f'\n═══ Split ═══')
print(f'Train (nur normal):    {len(df_train):>9,} | bis {train_end.date()}')
print(f'  davon demand=0:      {(df_train["total_demand"]==0).sum():>9,} '
      f'({(df_train["total_demand"]==0).mean():.1%})')
print(f'Val   (ohne Grauzone): {len(df_val):>9,} | '
      f'Anomalien: {(df_val["label"]=="anomal").sum():,} '
      f'({(df_val["label"]=="anomal").mean():.2%})')
print(f'Test  (ohne Grauzone): {len(df_test):>9,} | '
      f'Anomalien: {(df_test["label"]=="anomal").sum():,} '
      f'({(df_test["label"]=="anomal").mean():.2%})')

# ── Normalisierung (Scaler nur auf Train!) ──
scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_train[FEATURE_COLS].values)
val_scaled   = scaler.transform(df_val[FEATURE_COLS].values)
test_scaled  = scaler.transform(df_test[FEATURE_COLS].values)
print(f'\nScaler fitted auf {len(train_scaled):,} Trainings-Samples')

# Scaler-Statistik (für Nachvollziehbarkeit)
print(f'\nScaler-Means (ausgewählte Features):')
for i, col in enumerate(FEATURE_COLS[:6]):
    print(f'  {col}: mean={scaler.mean_[i]:.3f}, std={scaler.scale_[i]:.3f}')

# ── Sequenzen bilden (pro Station) ──
def create_station_sequences(df, scaled_data, window_size):
    """Erstellt Sequenzen der Länge window_size pro Station."""
    sequences, labels, meta = [], [], []
    station_ids = df['station_id'].values
    timestamps  = df['hour_ts'].values
    label_arr   = df['label'].values

    for station in df['station_id'].unique():
        mask = station_ids == station
        s_data   = scaled_data[mask]
        s_labels = label_arr[mask]
        s_times  = timestamps[mask]

        if len(s_data) < window_size:
            continue

        for i in range(len(s_data) - window_size + 1):
            sequences.append(s_data[i:i + window_size])
            labels.append(s_labels[i + window_size - 1])
            meta.append({
                'station_id': station,
                'timestamp': s_times[i + window_size - 1]
            })

    return np.array(sequences, dtype=np.float32), np.array(labels), meta

print(f'\nSequenzen erstellen (Window={WINDOW_SIZE})...')
X_train, y_train, meta_train = create_station_sequences(df_train, train_scaled, WINDOW_SIZE)
X_val,   y_val,   meta_val   = create_station_sequences(df_val,   val_scaled,   WINDOW_SIZE)
X_test,  y_test,  meta_test  = create_station_sequences(df_test,  test_scaled,  WINDOW_SIZE)

print(f'X_train: {X_train.shape}  (nur normal)')
print(f'X_val:   {X_val.shape}    (Anomalien: {(y_val=="anomal").sum()})')
print(f'X_test:  {X_test.shape}   (Anomalien: {(y_test=="anomal").sum()})')

# Für Vanilla AE und VAE: flatten
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f'Flattened dim: {INPUT_DIM_FLAT}')

# ── Sanity Checks ──
print(f'\n═══ SANITY CHECKS ═══')
print(f'y_train: {np.unique(y_train, return_counts=True)}')
print(f'y_val:   {np.unique(y_val, return_counts=True)}')
print(f'y_test:  {np.unique(y_test, return_counts=True)}')

# Kein z_score in den Features?
assert 'z_score' not in FEATURE_COLS, "FEHLER: z_score darf NICHT in FEATURE_COLS sein!"
assert 'demand_residual' not in FEATURE_COLS, "FEHLER: demand_residual darf NICHT in Features!"
assert 'demand_ratio' not in FEATURE_COLS, "FEHLER: demand_ratio darf NICHT in Features!"
assert (y_train == 'normal').all(), "FEHLER: y_train enthält nicht-normale Labels!"
assert (y_val == 'anomal').sum() > 0, "FEHLER: Keine Anomalien in Validation!"
assert (y_test == 'anomal').sum() > 0, "FEHLER: Keine Anomalien in Test!"
print('✅ Alle Checks bestanden — keine Leakage-Features, Labels korrekt.\n')


✅ Gespeichert: /content/data/V4_AD_Mannheim.parquet
   Shape: (2278748, 31)

═══ Split ═══
Train (nur normal):    1,455,707 | bis 2025-02-27
  davon demand=0:        850,397 (58.4%)
Val   (ohne Grauzone):   338,014 | Anomalien: 1,637 (0.48%)
Test  (ohne Grauzone):   366,999 | Anomalien: 1,217 (0.33%)

Scaler fitted auf 1,455,707 Trainings-Samples

Scaler-Means (ausgewählte Features):
  total_demand: mean=1.272, std=2.698
  n_lends: mean=0.635, std=1.520
  n_returns: mean=0.637, std=1.600
  demand_lag_1h: mean=1.442, std=3.049
  demand_lag_24h: mean=1.449, std=3.041
  demand_lag_168h: mean=1.454, std=3.051

Sequenzen erstellen (Window=24)...
X_train: (1453591, 24, 19)  (nur normal)
X_val:   (335898, 24, 19)    (Anomalien: 1633)
X_test:  (364883, 24, 19)   (Anomalien: 1211)
Flattened dim: 456

═══ SANITY CHECKS ═══
y_train: (array(['normal'], dtype='<U6'), array([1453591]))
y_val:   (array(['anomal', 'normal'], dtype='<U6'), array([  1633, 334265]))
y_test:  (array(['anomal', 'normal'],

In [6]:
# ══════════════════════════════════════════════════════════════
# 9 — Modellarchitekturen
# ══════════════════════════════════════════════════════════════

class VanillaAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def get_latent(self, x):
        return self.encoder(x)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=32, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(
            input_size=n_features, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.decoder = nn.LSTM(
            input_size=latent_dim, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.output_layer = nn.Linear(latent_dim, n_features)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, cell) = self.encoder(x)
        latent = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        decoder_out, _ = self.decoder(latent, (hidden, cell))
        return self.output_layer(decoder_out)

    def get_latent(self, x):
        _, (hidden, _) = self.encoder(x)
        return hidden[-1]


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu


# ══════════════════════════════════════════════════════════════
# 10 — Training Framework
# ══════════════════════════════════════════════════════════════

def train_ae(model, X_train, X_val, model_name='AE',
             epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE,
             is_vae=False, vae_beta=1.0):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=7
    )
    criterion = nn.MSELoss()
    scaler_amp = torch.amp.GradScaler('cuda')

    train_tensor = torch.FloatTensor(X_train).to(device)
    train_loader = DataLoader(
        TensorDataset(train_tensor, train_tensor),
        batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=False
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    EARLY_STOP_PATIENCE = 15

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                if is_vae:
                    x_hat, mu, logvar = model(batch_x)
                    recon_loss = criterion(x_hat, batch_x)
                    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = recon_loss + vae_beta * kl_loss
                else:
                    x_hat = model(batch_x)
                    loss = criterion(x_hat, batch_x)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Validation (alle 3 Epochs)
        if (epoch + 1) % 3 == 0 or epoch == 0 or epoch == epochs - 1:
            model.eval()
            val_losses = []
            with torch.no_grad(), torch.amp.autocast('cuda'):
                for vi in range(0, len(X_val), 4096):
                    v_chunk = torch.FloatTensor(X_val[vi:vi+4096]).to(device)
                    if is_vae:
                        v_hat, v_mu, v_lv = model(v_chunk)
                        v_recon = criterion(v_hat, v_chunk)
                        v_kl = -0.5 * torch.mean(1 + v_lv - v_mu.pow(2) - v_lv.exp())
                        val_losses.append((v_recon + vae_beta * v_kl).item())
                    else:
                        v_hat = model(v_chunk)
                        val_losses.append(criterion(v_hat, v_chunk).item())
                    del v_chunk, v_hat
            val_loss = np.mean(val_losses)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 3
        else:
            val_loss = history['val_loss'][-1] if history['val_loss'] else avg_train_loss

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  [{model_name}] Epoch {epoch+1:>3}/{epochs} | '
                  f'Train: {avg_train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr_now:.1e}')

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'  [{model_name}] Early stopping at epoch {epoch+1}')
            break

    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    print(f'  [{model_name}] Best Val Loss: {best_val_loss:.6f}\n')
    return history


# ══════════════════════════════════════════════════════════════
# 11 — Scoring & Evaluation
#
# [FIX #6] Scoring über GANZE Sequenz, nicht nur letzte N Stunden
# BEGRÜNDUNG: "Letzte 3h" war ein Hyperparameter, der auf das eigene
# Labeling-Schema optimiert. Die ganze Sequenz ist konservativer
# und ehrlicher. Der AE-Reconstruction-Error über 24h ist ein
# robusteres Anomalie-Signal.
#
# [FIX #4] Demand-only Scoring: Nur die 6 Demand-Features werden
# für den Score herangezogen. Wetter/Kalender sollen rekonstruiert
# werden (hilft dem AE), aber nicht in den Score einfließen.
# ══════════════════════════════════════════════════════════════

def compute_anomaly_scores(model, X, is_vae=False, chunk_size=2048,
                           demand_indices=DEMAND_FEATURE_INDICES):
    """
    Reconstruction Error NUR auf Demand-Features, über die GANZE Sequenz.
    """
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            if is_vae:
                x_hat, _, _ = model(chunk)
            else:
                x_hat = model(chunk)

            if chunk.dim() == 3:
                # LSTM: (batch, seq_len, features) → Demand-Features filtern
                diff = (chunk[:, :, demand_indices] - x_hat[:, :, demand_indices]) ** 2
            else:
                # Flat: (batch, seq_len*features) → reshape, filtern
                batch_size = chunk.shape[0]
                c_3d = chunk.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                h_3d = x_hat.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                diff = (c_3d[:, :, demand_indices] - h_3d[:, :, demand_indices]) ** 2

            # Mean über ganze Sequenz und alle Demand-Features
            scores = diff.mean(dim=tuple(range(1, diff.dim())))

        all_scores.append(scores.cpu())
        del chunk, x_hat, scores, diff
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()


def find_best_threshold(scores, labels):
    """Schwellenwert der den F1-Score auf Val maximiert."""
    binary = (labels == 'anomal').astype(int)
    if binary.sum() == 0:
        return np.percentile(scores, 95), 0.0
    prec, rec, thresholds = precision_recall_curve(binary, scores)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1[:-1])
    return thresholds[best_idx], f1[best_idx]


def evaluate_model(model, X, y, threshold, model_name='AE', is_vae=False):
    """Evaluation mit allen Metriken."""
    scores = compute_anomaly_scores(model, X, is_vae=is_vae)
    binary = (y == 'anomal').astype(int)
    preds  = (scores > threshold).astype(int)

    prec, rec, _ = precision_recall_curve(binary, scores)
    auc_pr = auc(rec, prec)
    f1 = f1_score(binary, preds, zero_division=0)

    try:
        auc_roc = roc_auc_score(binary, scores)
    except ValueError:
        auc_roc = 0.0

    print(f'── {model_name} auf Testdaten ──')
    print(f'  AUC-PR:  {auc_pr:.4f}  (Hauptmetrik)')
    print(f'  F1:      {f1:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  Threshold: {threshold:.6f}')
    print(classification_report(binary, preds, target_names=['Normal', 'Anomal'],
                                zero_division=0))

    return {
        'model': model_name, 'auc_pr': auc_pr, 'f1': f1, 'auc_roc': auc_roc,
        'threshold': threshold, 'scores': scores, 'preds': preds
    }

In [7]:
# ══════════════════════════════════════════════════════════════
# 12 — Training & Evaluation
# ══════════════════════════════════════════════════════════════
print('='*60)
print('MODELL 1: Vanilla Autoencoder (Baseline)')
print('='*60)
torch.cuda.empty_cache()

vanilla_ae = VanillaAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vanilla_ae.parameters()):,}')

hist_vanilla = train_ae(vanilla_ae, X_train_flat, X_val_flat,
                        model_name='VanillaAE', epochs=EPOCHS, batch_size=BATCH_SIZE)

scores_val_v = compute_anomaly_scores(vanilla_ae, X_val_flat)
thresh_v, f1_v = find_best_threshold(scores_val_v, y_val)
print(f'Val-Threshold: {thresh_v:.6f} (F1={f1_v:.4f})')
results_vanilla = evaluate_model(vanilla_ae, X_test_flat, y_test, thresh_v, 'VanillaAE')

vanilla_ae = vanilla_ae.cpu()
torch.cuda.empty_cache()
print('✅ Vanilla AE fertig\n')

MODELL 1: Vanilla Autoencoder (Baseline)
Parameter: 308,456


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3676/322799642.py", line 12, in <cell line: 0>
    hist_vanilla = train_ae(vanilla_ae, X_train_flat, X_val_flat,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3676/2001877486.py", line 121, in train_ae
    for batch_x, _ in train_loader:
                      ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 801, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py"


KeyboardInterrupt



In [9]:
print('='*60)
print('MODELL 2: LSTM-Autoencoder')
print('='*60)
torch.cuda.empty_cache()

lstm_ae = LSTMAutoencoder(n_features=N_FEATURES, latent_dim=LATENT_DIM, n_layers=2)
print(f'Parameter: {sum(p.numel() for p in lstm_ae.parameters()):,}')

hist_lstm = train_ae(lstm_ae, X_train, X_val,
                     model_name='LSTM-AE', epochs=EPOCHS, batch_size=2048)

scores_val_l = compute_anomaly_scores(lstm_ae, X_val)
thresh_l, f1_l = find_best_threshold(scores_val_l, y_val)
print(f'Val-Threshold: {thresh_l:.6f} (F1={f1_l:.4f})')
results_lstm = evaluate_model(lstm_ae, X_test, y_test, thresh_l, 'LSTM-AE')

lstm_ae = lstm_ae.cpu()
torch.cuda.empty_cache()
print('✅ LSTM-AE fertig\n')

MODELL 2: LSTM-Autoencoder
Parameter: 32,755
  [LSTM-AE] Epoch   1/50 | Train: 0.405385 | Val: 0.318011 | LR: 1.0e-03
  [LSTM-AE] Epoch   5/50 | Train: 0.177061 | Val: 0.261990 | LR: 1.0e-03
  [LSTM-AE] Epoch  10/50 | Train: 0.145926 | Val: 0.225534 | LR: 1.0e-03
  [LSTM-AE] Epoch  15/50 | Train: 0.133426 | Val: 0.213579 | LR: 1.0e-03
  [LSTM-AE] Epoch  20/50 | Train: 0.126505 | Val: 0.207004 | LR: 1.0e-03
  [LSTM-AE] Epoch  25/50 | Train: 0.121313 | Val: 0.184115 | LR: 1.0e-03
  [LSTM-AE] Epoch  30/50 | Train: 0.117203 | Val: 0.176962 | LR: 1.0e-03
  [LSTM-AE] Epoch  35/50 | Train: 0.113981 | Val: 0.185022 | LR: 1.0e-03
  [LSTM-AE] Epoch  40/50 | Train: 0.110783 | Val: 0.172781 | LR: 1.0e-03
  [LSTM-AE] Epoch  45/50 | Train: 0.108364 | Val: 0.169672 | LR: 1.0e-03
  [LSTM-AE] Epoch  50/50 | Train: 0.106888 | Val: 0.167210 | LR: 1.0e-03
  [LSTM-AE] Best Val Loss: 0.167210

Val-Threshold: 1.593340 (F1=0.0575)
── LSTM-AE auf Testdaten ──
  AUC-PR:  0.0306  (Hauptmetrik)
  F1:      0.0613


In [8]:

print('='*60)
print('MODELL 3: Variational Autoencoder (VAE)')
print('='*60)
torch.cuda.empty_cache()

vae = VAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vae.parameters()):,}')

hist_vae = train_ae(vae, X_train_flat, X_val_flat,
                    model_name='VAE', epochs=EPOCHS, batch_size=BATCH_SIZE,
                    is_vae=True, vae_beta=0.5)

scores_val_vae = compute_anomaly_scores(vae, X_val_flat, is_vae=True)
thresh_vae, f1_vae = find_best_threshold(scores_val_vae, y_val)
print(f'Val-Threshold: {thresh_vae:.6f} (F1={f1_vae:.4f})')
results_vae = evaluate_model(vae, X_test_flat, y_test, thresh_vae, 'VAE', is_vae=True)

vae = vae.cpu()
torch.cuda.empty_cache()
print('✅ VAE fertig\n')

MODELL 3: Variational Autoencoder (VAE)
Parameter: 312,584
  [VAE] Epoch   1/50 | Train: 0.741649 | Val: 0.615991 | LR: 1.0e-03
  [VAE] Epoch   5/50 | Train: 0.460247 | Val: 0.551587 | LR: 1.0e-03
  [VAE] Epoch  10/50 | Train: 0.447979 | Val: 0.529127 | LR: 1.0e-03
  [VAE] Epoch  15/50 | Train: 0.442639 | Val: 0.522465 | LR: 1.0e-03
  [VAE] Epoch  20/50 | Train: 0.439293 | Val: 0.520315 | LR: 1.0e-03
  [VAE] Epoch  25/50 | Train: 0.436921 | Val: 0.517912 | LR: 1.0e-03
  [VAE] Epoch  30/50 | Train: 0.435013 | Val: 0.515756 | LR: 1.0e-03
  [VAE] Epoch  35/50 | Train: 0.433590 | Val: 0.515293 | LR: 1.0e-03
  [VAE] Epoch  40/50 | Train: 0.432690 | Val: 0.513759 | LR: 1.0e-03
  [VAE] Epoch  45/50 | Train: 0.431726 | Val: 0.513946 | LR: 1.0e-03
  [VAE] Epoch  50/50 | Train: 0.429600 | Val: 0.512737 | LR: 5.0e-04
  [VAE] Best Val Loss: 0.512335

Val-Threshold: 2.669300 (F1=0.0611)
── VAE auf Testdaten ──
  AUC-PR:  0.0311  (Hauptmetrik)
  F1:      0.0642
  AUC-ROC: 0.9187
  Threshold: 2.66930

In [ ]:










# ══════════════════════════════════════════════════════════════
# 13 — Vergleich
# ══════════════════════════════════════════════════════════════

print('='*60)
print('MODELLVERGLEICH (V4 — Saubere Pipeline, ohne Leakage)')
print('='*60)
print(f'{"Modell":<12} {"AUC-PR":>8} {"F1":>8} {"AUC-ROC":>8}')
print('-'*40)
for r in [results_vanilla, results_lstm, results_vae]:
    print(f'{r["model"]:<12} {r["auc_pr"]:>8.4f} {r["f1"]:>8.4f} {r["auc_roc"]:>8.4f}')


# ══════════════════════════════════════════════════════════════
# 14 — Diagnose-Plots
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('V4 — Saubere AD Pipeline (ohne Leakage)', fontsize=14, fontweight='bold')

for idx, (results, name) in enumerate([
    (results_vanilla, 'VanillaAE'),
    (results_lstm, 'LSTM-AE'),
    (results_vae, 'VAE')
]):
    scores = results['scores']
    binary = (y_test == 'anomal').astype(int)

    # Score-Verteilung
    ax = axes[0, idx]
    ax.hist(scores[binary == 0], bins=100, alpha=0.7, label='Normal', density=True, color='steelblue')
    ax.hist(scores[binary == 1], bins=100, alpha=0.7, label='Anomal', density=True, color='crimson')
    ax.axvline(results['threshold'], color='k', linestyle='--', label=f'Thresh={results["threshold"]:.3f}')
    ax.set_title(f'{name} — Score-Verteilung')
    ax.set_xlabel('Reconstruction Error (Demand-only)')
    ax.legend()
    ax.set_yscale('log')

    # PR Curve
    ax = axes[1, idx]
    prec, rec, _ = precision_recall_curve(binary, scores)
    ax.plot(rec, prec, color='darkorange', lw=2)
    ax.set_title(f'{name} — PR Curve (AUC={results["auc_pr"]:.4f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/data/V4_diagnosis_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Diagnose-Plots gespeichert.')


# ══════════════════════════════════════════════════════════════
# 15 — Modelle & Artefakte speichern (Transfer-Ready)
# ══════════════════════════════════════════════════════════════

save_dir = '/content/data/models_v4'
os.makedirs(save_dir, exist_ok=True)

torch.save(vanilla_ae.state_dict(), f'{save_dir}/v4_vanilla_ae_mannheim.pth')
torch.save(lstm_ae.state_dict(),    f'{save_dir}/v4_lstm_ae_mannheim.pth')
torch.save(vae.state_dict(),        f'{save_dir}/v4_vae_mannheim.pth')

import pickle

config = {
    # Pipeline-Konfiguration (für Reproduzierbarkeit)
    'FEATURE_COLS': FEATURE_COLS,
    'DEMAND_FEATURE_INDICES': DEMAND_FEATURE_INDICES,
    'WINDOW_SIZE': WINDOW_SIZE,
    'LATENT_DIM': LATENT_DIM,
    'N_FEATURES': N_FEATURES,
    'INPUT_DIM_FLAT': INPUT_DIM_FLAT,
    # Labeling-Konfiguration
    'Z_ANOMALY_THRESHOLD': Z_ANOMALY_THRESHOLD,
    'Z_TRAIN_THRESHOLD': Z_TRAIN_THRESHOLD,
    'MIN_HIST_MEAN': MIN_HIST_MEAN,
    'MIN_ABSOLUTE': MIN_ABSOLUTE,
    'MIN_EVENTS_PER_DAY': MIN_EVENTS_PER_DAY,
    # Thresholds (aus Validation)
    'thresholds': {
        'vanilla_ae': thresh_v,
        'lstm_ae': thresh_l,
        'vae': thresh_vae,
    },
    # Ergebnisse (ohne Scores/Preds wegen Speicher)
    'results': {
        name: {k: v for k, v in r.items() if k not in ('scores', 'preds')}
        for name, r in [('vanilla_ae', results_vanilla),
                        ('lstm_ae', results_lstm),
                        ('vae', results_vae)]
    },
    # Source City Metadaten (für Transfer)
    'source_city': CITY,
    'source_stations': hourly_full['station_id'].nunique(),
    'source_period': f'{t_min.date()} – {t_max.date()}',
    'source_anomaly_rate': is_anomaly.mean(),
}

with open(f'{save_dir}/v4_config.pkl', 'wb') as f:
    pickle.dump(config, f)

with open(f'{save_dir}/v4_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'\n✅ Alles gespeichert in {save_dir}/')
print('   → Modelle, Scaler und Config bereit für Cross-City Transfer.')
print(f'\n   WICHTIG für Transfer:')
print(f'   - Scaler wird in Target City NEU gefittet (nicht übertragen)')
print(f'   - Labeling-Stats werden in Target City NEU berechnet')
print(f'   - Nur die Modellgewichte werden transferiert (Pre-Train/Fine-Tune)')